# TMol How-To Guide — Advanced Topics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/master/docs/tutorial/08_advanced_topics.ipynb)

## 8. Advanced Topics

Two more advanced topics: a) _designing_ sequences using tmol's `pack_rotamers`, and b) a few examples illustrating how to use tmol _inside_ a machine-learning model, where gradients through tmol need to survive backpropagation.

### Setup

Wheel install, imports, seeding, and device selection live in [notebook_setup.py](https://github.com/uw-ipd/tmol/blob/master/docs/tutorial/notebook_setup.py).

In [1]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlretrieve

    urlretrieve(
        "https://raw.githubusercontent.com/uw-ipd/tmol/"
        "master/docs/tutorial/notebook_setup.py",
        "notebook_setup.py",
    )

from notebook_setup import *  # noqa: F401,F403

device, param_db = setup()

W1021 18:53:42.374000 109766 site-packages/torch/utils/cpp_extension.py:183] No CUDA runtime is found, using CUDA_HOME='/home/jflat06/miniconda3/envs/tmol'


### Design with pack_rotamers

The tmol function `pack_rotamers` carries out discrete optimization of sidechain identities and conformations, with per-amino-acid options controlled by a `PackerTask`.  We have previously only seen the packer used in conjunction with `restrict_to_repacking`, which only samples conformations of the input amino acids.  However, a `PackerTask` without this option set can be used to change residue identities.

The block below shows how the packer may be used for sequence redesign, by redesigning the core of ubiquitin to all hydrophobic residues.  Several new concepts are introduced:
- Showing per-residue control of a `PackerTask`
- `restrict_absent_name3s` to offer more control over design, restricting to a subset of all amino acids

#### Redesigning the core

In [2]:
cif_text = fetch_cif("1UBQ")
pose_stack = pose_stack_from_file(io.StringIO(cif_text), device, param_db=param_db)
score_function = beta2016_score_function(device, param_db=param_db)

# ubiquitin's buried core, by PDB residue number
core_mask = torch.zeros((1, pose_stack.max_n_blocks), dtype=torch.bool, device=device)
core_residues = torch.tensor([3, 5, 13, 15, 17, 26, 30, 43, 50, 56, 61, 67])
core_mask[0, tmol.blocks_for_residue_labels(pose_stack, 0, "A", core_residues)] = True
samplers = [create_dunbrack_sampler_from_database(param_db, device), FixedAAChiSampler(), IncludeCurrentSampler()]


def design_core(task):
    """Task operation: design the core to hydrophobics and repack everything else."""
    task.restrict_to_repacking(~core_mask)
    task.restrict_absent_name3s({"ALA", "VAL", "LEU", "ILE", "PHE", "MET"}, core_mask)
    for sampler in samplers:
        task.add_conformer_sampler(sampler)


def design_summary(designed, mask):
    """Total score and the mutations within mask, e.g. ILE13VAL."""
    residue_numbers = pose_stack.pdb_info.residue_labels[0]
    mutations = [
        f"{pose_stack.block_type(0, block).name3}{residue_numbers[block]}{designed.block_type(0, block).name3}"
        for block in torch.nonzero(mask[0]).flatten().tolist()
        if designed.block_type(0, block).name3 != pose_stack.block_type(0, block).name3
    ]
    score = float(score_function.render_whole_pose_scoring_module(designed)(designed.coords).sum())
    return f"score {score:.1f}; mutations: {', '.join(mutations) or 'none'}"


task = PackerTask(pose_stack, PackerPalette())
design_core(task)
torch.manual_seed(SEED)
packer_design = pack_rotamers(pose_stack, score_function, task)

core_notes = {"original": design_summary(pose_stack, core_mask), "packer design": design_summary(packer_design, core_mask)}
for label, note in core_notes.items():
    print(f"{label}: {note}")

display(tmol.overlay_view({"original": pose_stack, "packer design": packer_design}, notes=core_notes))

original: score 201.5; mutations: none
packer design: score -90.3; mutations: ILE13VAL, ILE61VAL


#### Design inside relax

A `PackerTask` allowing design can also be passed into `fast_relax`.  This results in a design pipeline alternating sequence design and all-atom minimization.  Note that the reported relaxed score also includes _minimization_, so it is not directly comparable to the packer-only score above.

In [3]:
# the same task operation, applied at every packing step of relax
torch.manual_seed(SEED)
relax_design = tmol.fast_relax(
    pose_stack.clone(), score_function, PackerPalette(), tmol.CartesianMoveMap(),
    tmol.FoldForest.reasonable_fold_forest(pose_stack), num_repeats=1, task_operations=[design_core],
)

core_notes["design in relax"] = design_summary(relax_design, core_mask)
print(f"design in relax: {core_notes['design in relax']}")

display(tmol.overlay_view(
    {"original": pose_stack, "packer design": packer_design, "design in relax": relax_design}, notes=core_notes
))

design in relax: score -259.9; mutations: VAL5LEU, ILE13VAL, VAL17MET, ILE30ALA


#### Design with noncanonical residues

The `PackerPalette` decides which residue types are candidates at all, before the `PackerTask` further restricts candidates. The default palette offers only canonical L residues (plus glycine); `PackerPalette(noncanonical_design=True)` also includes AIB (alpha-aminoisobutyric acid) and D-amino acids. The following example shows design of several solvent-exposed helical positions, allowing both AIB and D-ALA in addition to the original identity.

In [4]:
# solvent-facing positions on ubiquitin's helix (residues 23-34)
helix_mask = torch.zeros_like(core_mask)
helix_blocks = tmol.blocks_for_residue_labels(pose_stack, 0, "A", torch.tensor([25, 28, 29, 32, 33]))
helix_mask[0, helix_blocks] = True

# the default palette offers only canonical L residues; noncanonical_design adds the rest
pbt = pose_stack.packed_block_types
offered = {}
for palette in (PackerPalette(), PackerPalette(noncanonical_design=True)):
    palette_task = PackerTask(pose_stack, palette)
    considered = palette_task.per_block_considered_block_types[0, int(helix_blocks[0])].tolist()
    offered[palette.noncanonical_design] = {pbt.active_block_types[i].name3 for i in considered if i >= 0}
print("noncanonical_design=True also offers:", sorted(offered[True] - offered[False]))

# offer D-alanine (DAL), AIB, or the native residue at each helix position
helix_task = PackerTask(pose_stack, PackerPalette(noncanonical_design=True))
helix_task.restrict_to_repacking(~helix_mask)
for block in helix_blocks.tolist():
    block_mask = torch.zeros_like(helix_mask)
    block_mask[0, block] = True
    helix_task.restrict_absent_name3s({"DAL", "AIB", pose_stack.block_type(0, block).name3}, block_mask)
for sampler in samplers:
    helix_task.add_conformer_sampler(sampler)

torch.manual_seed(SEED)
helix_design = pack_rotamers(pose_stack, score_function, helix_task)

helix_notes = {"original": design_summary(pose_stack, helix_mask), "helix design": design_summary(helix_design, helix_mask)}
for label, note in helix_notes.items():
    print(f"{label}: {note}")

display(tmol.overlay_view({"original": pose_stack, "helix design": helix_design}, notes=helix_notes))

noncanonical_design=True also offers: ['AIB', 'DAL', 'DAR', 'DAS', 'DCY', 'DGL', 'DGN', 'DHI', 'DIL', 'DLE', 'DLY', 'DPN', 'DPR', 'DSG', 'DSN', 'DTH', 'DTR', 'DTY', 'DVA', 'MED']


original: score 201.5; mutations: none
helix design: score -90.5; mutations: ASN25AIB, ASP32AIB


### tmol as a differentiable loss

One advantage of tmol is that every step from an input coordinate (or internal coordinate) tensor to score is a torch operation. A model that emits coordinates can therefore use tmol energy as a loss term, and backpropogation (via `.backward()`) works through tmol like any other torch module.  Support varies depending on the input coordinate representation:

1. **atom37**, an `Ntokenx37x3`tensor format used by AtomWorks, is supported directly.
2. **Any other layout** needs a small interface layer that routes the model's coordinates into tmol's atoms.

Examples of both are shown below.

#### Mode 1: atom37 -- a tmol loss for a structure refiner

`pose_stack_from_canonical_aa_atom37` builds poses from atom37 coordinates, AtomWorks residue tokens and chain ids alone; `atomworks_from_pose_stack` goes the other way and is used here to make training targets. Token 0 marks padding, so proteins of different lengths share one batch. (For noncanonical residues or ligands in an atom37-like layout, `pose_stack_from_atom37` takes the same tensors plus a `PoseBuildContext` that supplies the chemistry.)

In [5]:
from tmol.io import ATOMWORKS_ATOM37_NAMES, atomworks_from_pose_stack, pose_stack_from_canonical_aa_atom37

# training targets: ubiquitin and Trp-cage as one atom37 batch; the shorter one is padded with token 0
targets = PoseStackBuilder.from_poses(
    [pose_stack_from_file(io.StringIO(fetch_cif(pdb_id)), device, param_db=param_db) for pdb_id in ("1UBQ", "1L2Y")],
    device,
)
true_coords, residue_type, chain_iid = atomworks_from_pose_stack(targets)
present = (true_coords != 0).any(dim=-1, keepdim=True)  # absent atom37 slots are 0
n_residues = int((residue_type > 0).sum())
print(f"atom37 batch {tuple(true_coords.shape)}, real residues per pose {(residue_type > 0).sum(dim=1).tolist()}")

atom37_score_function = beta2016_score_function(device)


def atom37_energies(coords, sum_terms=True):
    """Build poses from atom37 tensors and score them, keeping the autograd graph."""
    pose_stack = pose_stack_from_canonical_aa_atom37(coords, residue_type, chain_iid)
    scorer = atom37_score_function.render_whole_pose_scoring_module(pose_stack)
    return scorer(pose_stack.coords, sum_terms=sum_terms)

ASN is seen only at a chain terminus and N, ND2 could each carry its other connection; it is read as the conventional backbone through N. A copy of the residue in a chain would settle it.


atom37 batch (2, 76, 37, 3), real residues per pose [76, 20]


The model is a deliberately small refiner: for each residue, an MLP reads the atoms of residues i-1, i and i+1 (relative to residue i's CA) plus a residue-type embedding, and predicts a correction to residue i's atoms. It learns to clean up noisy coordinates.

In [6]:
ca_slot = ATOMWORKS_ATOM37_NAMES["ALA"].index("CA")


class NeighborRefiner(torch.nn.Module):
    """Predict a correction for each atom37 slot from residues i-1, i, i+1, centered on CA_i."""

    def __init__(self, hidden=256):
        super().__init__()
        self.embed = torch.nn.Embedding(21, 16)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(3 * 37 * 3 + 16, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, 37 * 3),
        )

    def forward(self, coords):
        n_tokens = coords.shape[1]
        position = torch.arange(n_tokens, device=coords.device)
        ca = coords[:, :, ca_slot : ca_slot + 1]
        features = [self.embed(residue_type)]
        for shift in (1, 0, -1):
            # a neighbor counts only if it exists, is a real residue, and is in the same chain
            in_range = (position - shift >= 0) & (position - shift < n_tokens)
            real_neighbor = (chain_iid.roll(shift, 1) == chain_iid) & (residue_type.roll(shift, 1) > 0) & in_range
            keep = present.roll(shift, 1) & real_neighbor[..., None, None]
            features.append(((coords.roll(shift, 1) - ca) * keep).flatten(2))
        correction = self.mlp(torch.cat(features, dim=-1)).view_as(coords)
        return coords + correction * present


def noisy(generator, sigma=0.3):
    """The true coordinates with Gaussian noise (sigma in A) on every present atom."""
    noise = torch.randn(true_coords.shape, generator=generator, device=device)
    return true_coords + sigma * noise * present


def rmsd(coords):
    """RMSD to the true coordinates over present atoms, in A."""
    squared = ((coords - true_coords) ** 2).sum(dim=-1, keepdim=True) * present
    return float((squared.sum() / present.sum()).sqrt())


def fit(model, steps, energy_weight, lr, seed):
    """Train on fresh noise each step; loss = MSE + energy_weight * tmol energy per residue."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    generator = torch.Generator(device=device).manual_seed(seed)
    for _ in range(steps):
        predicted = model(noisy(generator))
        loss = (((predicted - true_coords) ** 2) * present).sum() / present.sum()
        if energy_weight:
            loss = loss + energy_weight * atom37_energies(predicted).sum() / n_residues
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

Coordinate MSE alone already pulls the model toward the true structure, so a tmol term helps most once MSE has done what it can: the remaining errors are geometry that MSE averages away. The realistic recipe is to pretrain on MSE and then fine-tune with the energy added. Below, a pretrained refiner is fine-tuned two ways for the same number of steps -- more MSE, or MSE plus 0.01 x tmol energy per residue -- and both are compared on noise neither saw.

In [7]:
import copy

# 1) pretrain on MSE alone
torch.manual_seed(SEED)
pretrained = fit(NeighborRefiner(), steps=2000, energy_weight=0.0, lr=1e-3, seed=SEED)

# 2) fine-tune two copies on the same data: more MSE, or MSE plus the tmol energy
refiners = {
    "MSE": fit(copy.deepcopy(pretrained), steps=500, energy_weight=0.0, lr=3e-4, seed=SEED + 1),
    "MSE + tmol": fit(copy.deepcopy(pretrained), steps=500, energy_weight=0.01, lr=3e-4, seed=SEED + 1),
}

# 3) compare on noise neither model saw
eval_input = noisy(torch.Generator(device=device).manual_seed(SEED + 2))
term_names = [score_type.name for score_type in atom37_score_function.all_score_types()]
outputs, notes = {"true": true_coords, "noisy input": eval_input}, {}
with torch.no_grad():
    outputs.update({label: refiner(eval_input) for label, refiner in refiners.items()})
    for label, coords in outputs.items():
        terms = atom37_energies(coords, sum_terms=False).sum(dim=-1)
        bonded = sum(float(terms[term_names.index(name)]) for name in ("cart_lengths", "cart_angles"))
        notes[label] = f"RMSD {rmsd(coords):.2f} A, energy {float(terms.sum()):.1f}, bond lengths + angles {bonded:.1f}"
        print(f"{label:12s} {notes[label]}")

    # ubiquitin (pose 0) from each model
    views = {
        label: pose_stack_from_canonical_aa_atom37(coords[:1], residue_type[:1], chain_iid[:1])
        for label, coords in outputs.items()
    }
display(tmol.overlay_view(views, notes=notes))

true         RMSD 0.00 A, energy 306.9, bond lengths + angles 130.7
noisy input  RMSD 0.50 A, energy 21581.7, bond lengths + angles 16359.7
MSE          RMSD 0.19 A, energy 1166.8, bond lengths + angles 698.9
MSE + tmol   RMSD 0.20 A, energy 764.1, bond lengths + angles 614.5


#### Mode 2: a custom layout -- all-atom output of a co-folding model

Co-folding models in the AlphaFold3 family emit one flat `[n_atoms, 3]` tensor covering protein and ligand atoms together, which atom37 cannot represent. The interface layer below is written once per topology: it assigns each atom a token and a slot (one token per polymer residue, one per ligand atom, as AF3 tokenizes), and `prepare_atom37_pose_builder` binds that routing to tmol's chemistry, ligand parameters included. Each call is then a differentiable scatter and pose construction.

In [8]:
from tmol.io import atom_array_from_file, build_context_from_atom_array, prepare_atom37_pose_builder


class FlatAtomLayer:
    """Route a co-folding model's flat [n_atoms, 3] output into a PoseStack.

    Each atom of the model's topology gets a token (one per polymer residue, one per
    ligand atom, as AF3 tokenizes) and a slot within its token. The prepared builder
    binds that routing to tmol's chemistry once; each call is then a differentiable
    scatter plus pose construction.
    """

    def __init__(self, structure, device, param_db):
        structure = structure.copy()
        new_residue = np.r_[
            True,
            (structure.res_id[1:] != structure.res_id[:-1]) | (structure.chain_id[1:] != structure.chain_id[:-1]),
        ]
        new_token = new_residue | structure.hetero
        token_id = np.cumsum(new_token) - 1
        slot = np.arange(len(token_id)) - np.flatnonzero(new_token)[token_id]
        assert slot.max() < 37, "every token must fit in 37 slots"
        structure.set_annotation("token_id", token_id)
        structure.set_annotation("atom37_slot", slot)

        self.context = build_context_from_atom_array(
            structure, device, prepare_ligands=True, param_db=param_db, ligand_seed=SEED
        )
        self.builder = prepare_atom37_pose_builder(structure, self.context)
        self.n_tokens = int(token_id[-1]) + 1
        self.index = (
            torch.zeros(len(token_id), dtype=torch.int64, device=device),
            torch.as_tensor(token_id, device=device),
            torch.as_tensor(slot, device=device),
        )

    def __call__(self, flat_coords):
        """PoseStack from [n_atoms, 3] coordinates; gradients flow back to flat_coords."""
        atom37 = torch.full((1, self.n_tokens, 37, 3), float("nan"), device=flat_coords.device)
        return self.builder(atom37.index_put(self.index, flat_coords))


# the model's topology: heavy atoms of the 1STP complex, as a co-folding model would emit them
complex_structure = atom_array_from_file(io.StringIO(fetch_cif("1STP")))
complex_structure = complex_structure[(complex_structure.element != "H") & (complex_structure.res_name != "HOH")]
layer = FlatAtomLayer(complex_structure, device, param_db)
print(f"{complex_structure.array_length()} atoms in {layer.n_tokens} tokens")

*** Open Babel Warning  in WriteMolecule
  No 2D or 3D coordinates exist. Stereochemical information will be stored using an Open Babel extension. To generate 2D or 3D coordinates instead use --gen2D or --gen3D.


917 atoms in 137 tokens


As a use, take a flawed prediction of the streptavidin-biotin complex, with the ligand shifted into clashes, and refine it by gradient descent on the ligand's flat coordinates, with tmol energy as the objective. The same `complex_energy` could equally be a loss term while training the co-folding model.

In [9]:
complex_score_function = beta2016_score_function(device, param_db=layer.context.parameter_database)


def complex_energy(flat_coords):
    """tmol energy of the complex, differentiable in the flat coordinates."""
    pose_stack = layer(flat_coords)
    return complex_score_function.render_whole_pose_scoring_module(pose_stack)(pose_stack.coords).sum()


true_flat = torch.tensor(complex_structure.coord, dtype=torch.float32, device=device)
is_ligand = torch.as_tensor(complex_structure.hetero, device=device)


def ligand_rmsd(flat_coords):
    """Ligand RMSD to the crystal pose, in A."""
    return float(((flat_coords[is_ligand] - true_flat[is_ligand]) ** 2).sum(dim=-1).mean().sqrt())


# a flawed prediction: biotin shifted 1.5 A into clashes with the protein, plus 0.2 A noise
generator = torch.Generator(device=device).manual_seed(SEED)
predicted = true_flat.clone()
shift = 1.5 * torch.tensor([1.0, 1.0, 0.0], device=device) / 2**0.5
predicted[is_ligand] += shift + 0.2 * torch.randn((int(is_ligand.sum()), 3), generator=generator, device=device)

# refine by gradient descent on the ligand's flat coordinates; the protein stays fixed
ligand_xyz = predicted[is_ligand].clone().requires_grad_(True)
optimizer = torch.optim.Adam([ligand_xyz], lr=0.05)
print(f"crystal: energy {float(complex_energy(true_flat)):.1f}")
for step in range(151):
    flat = predicted.clone()
    flat[is_ligand] = ligand_xyz
    energy = complex_energy(flat)
    if step % 50 == 0:
        print(f"step {step:3d}: energy {float(energy):.1f}, ligand RMSD {ligand_rmsd(flat.detach()):.2f} A")
    optimizer.zero_grad()
    energy.backward()
    optimizer.step()

refined = predicted.clone()
refined[is_ligand] = ligand_xyz.detach()
display(tmol.overlay_view(
    {"crystal": layer(true_flat), "prediction": layer(predicted), "refined": layer(refined)},
    notes={
        label: f"ligand RMSD {ligand_rmsd(coords):.2f} A"
        for label, coords in [("crystal", true_flat), ("prediction", predicted), ("refined", refined)]
    },
))

crystal: energy 366.1
step   0: energy 947.0, ligand RMSD 1.51 A


/tmp/ipykernel_109766/360754011.py:34: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  print(f"step {step:3d}: energy {float(energy):.1f}, ligand RMSD {ligand_rmsd(flat.detach()):.2f} A")


step  50: energy 363.6, ligand RMSD 0.63 A


step 100: energy 348.3, ligand RMSD 0.45 A


step 150: energy 343.3, ligand RMSD 0.56 A
